## Step 1 — Install Dependencies

In [1]:
import sys, subprocess, os

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'seleniumbase', 'beautifulsoup4', 'requests', 'lxml', 'nest_asyncio',
], check=True)

subprocess.run(['apt-get', 'install', '-yqq', 'xvfb'], check=True)

def install_chrome() -> str:
    for binary in ['/usr/bin/google-chrome', '/usr/bin/chromium-browser']:
        if os.path.exists(binary):
            print(f'✅ Chrome already installed: {binary}')
            return binary
    print('Installing Google Chrome...')
    subprocess.run(['wget', '-q', '-O', '/tmp/chrome.deb',
        'https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb'], check=True)
    subprocess.run(['apt', 'install', '-yqq', '/tmp/chrome.deb'], check=True)
    if not os.path.exists('/usr/bin/google-chrome'):
        raise RuntimeError('Chrome install failed')
    return '/usr/bin/google-chrome'

CHROME_BINARY = install_chrome()
ver = subprocess.check_output([CHROME_BINARY, '--version']).decode().strip()
print(f'✅ {ver}')
print('✅ All dependencies ready')

Installing Google Chrome...
✅ Google Chrome 148.0.7778.178
✅ All dependencies ready


In [3]:
from google.colab import auth
auth.authenticate_user()

## Step 2 — Imports

In [4]:
from __future__ import annotations

import asyncio
import nest_asyncio
nest_asyncio.apply()

import copy
import json
import os
import random
import re
import time
from datetime import datetime
from pathlib import Path
from typing import Any
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from seleniumbase import SB

print('✅ All imports OK')

✅ All imports OK


## Step 3 — Configuration
> Edit `START_PAGE` / `END_PAGE` before running.

In [23]:
SITE_BASE = 'https://www.cars.com'
RESULTS_URL_TMPL = (
    'https://www.cars.com/shopping/results/'
    '?deal_ratings%5B%5D=good&zip=60606'
    '&maximum_distance=9999&sort=best_match_desc&page={page}'
)
USER_AGENT = (
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
)

START_PAGE = 41
END_PAGE   = 50

LINK_FOLDER  = 'car_links'
RAW_DATA_DIR = 'raw_data'
IMG_BASE_DIR = 'downloaded_images'

GCS_BUCKET = 'bronze-car-recsys'
GCS_PREFIX = 'raw_data'

print(f'Crawl range: pages {START_PAGE} → {END_PAGE}')

Crawl range: pages 41 → 50


## Step 4 — Utility Helpers

In [6]:
def text(el, default: Any = None, strip: bool = True) -> Any:
    if el is None: return default
    try: return el.get_text(strip=strip)
    except: return default

def attr(el, name: str, default: Any = None) -> Any:
    if el is None or not hasattr(el, 'get'): return default
    try: return el.get(name)
    except: return default

def to_int(raw: str | None) -> int | None:
    if not isinstance(raw, str) or not raw: return None
    cleaned = re.sub(r'\D', '', raw)
    return int(cleaned) if cleaned else None

def absolute_url(href: str | None) -> str | None:
    return urljoin(SITE_BASE, href) if href else None

def download_images(urls: list[str], folder: Path) -> None:
    if not urls: return
    folder.mkdir(parents=True, exist_ok=True)
    for i, url in enumerate(urls, 1):
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                (folder / f'{i}.jpg').write_bytes(resp.content)
        except Exception as e:
            print(f'  [img] {e}')

print('✅ Helpers defined')

✅ Helpers defined


## Step 5 — Browser Core (SeleniumBase UC + CDP)
> Uses `activate_cdp_mode` + `solve_captcha` — the only reliable free method
> against Cloudflare Turnstile in 2026.

In [7]:
def scroll_to_bottom(sb, pause: float = 0.3, max_rounds: int = 8) -> None:
    """Scroll safely — skip if page has no body."""
    try:
        if not sb.execute_script("return document.body !== null;"):
            return
        last = sb.execute_script("return document.body.scrollHeight;")
        for _ in range(max_rounds):
            sb.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            sb.sleep(pause)
            new = sb.execute_script("return document.body.scrollHeight;")
            if new == last:
                break
            last = new
    except Exception as e:
        print(f'  [scroll] {e}')


def get_soup(
    sb,
    url: str,
    target_css: str = 'body',
    scroll: bool = False,
) -> BeautifulSoup:
    """
    Navigate using uc_open_with_reconnect (no async, no event loop conflict).
    Disconnects ChromeDriver during page load so Cloudflare can't fingerprint it.
    Auto-solves Turnstile if it appears.
    """
    sb.uc_open_with_reconnect(url, reconnect_time=4)

    title = sb.get_title()
    if 'Just a moment' in title or 'Verifying' in title:
        try:
            print('  [captcha] Turnstile detected — solving...')
            sb.uc_gui_click_captcha()
            sb.sleep(3)
            print('  [captcha] ✅ Solved')
        except Exception as e:
            print(f'  [captcha] {e}')

    if target_css != 'body':
        try:
            sb.wait_for_element(target_css, timeout=12)
        except Exception:
            pass

    if scroll:
        scroll_to_bottom(sb)

    return BeautifulSoup(sb.get_page_source(), 'html.parser')


print('✅ get_soup / scroll_to_bottom defined')

✅ get_soup / scroll_to_bottom defined


## Step 6 — Parsers

### 6a — Post (listing data)

In [8]:
def _parse_post(soup: BeautifulSoup) -> dict:
    post: dict = {}

    title_raw = text(soup.select_one('#vehicle-title'))
    if title_raw:
        parts = title_raw.split(' ', 1)
        post['new_used'] = parts[0]
        post['title']    = parts[1] if len(parts) > 1 else None
    else:
        post['new_used'] = post['title'] = None

    post['mileage']         = to_int(text(soup.select_one('div.msrp')))
    post['price']           = to_int(text(soup.select_one('.list-price')))
    post['monthly_payment'] = to_int(
        text(soup.select_one('fuse-stack[data-qa="monthly-payment"] fuse-button'))
    )

    basics: dict = {}
    subtitle = text(soup.select_one('section#features-and-specs div.subtitle'), default='')
    for pattern, key in [(r'VIN:\s*([A-Z0-9]+)', 'VIN'), (r'Stock #:\s*([A-Z0-9]+)', 'Stock Number')]:
        m = re.search(pattern, subtitle)
        if m: basics[key] = m.group(1)

    for entry in soup.select('div.basics li[data-qa="basics-entry"]'):
        raw = text(entry)
        if not raw: continue
        m = re.search(
            r'(?i)(.*)\s+(exterior color|interior color|fuel type|engine|mpg|drivetrain|transmission)$', raw)
        if m:
            val, k = m.group(1).strip(), m.group(2).strip().title()
            basics['MPG' if k.lower() == 'mpg' else k] = val
    post['basic_desc'] = basics or None

    features: dict = {}
    for block in soup.select('div.highlight-feature'):
        k = text(block.select_one('h3.features-spec-heading'))
        if k:
            features[k] = [text(li) for li in block.select('li[data-qa="spec-value"]')]
    post['feature_des'] = features or None

    history: dict = {}
    for span in soup.select('section#vehicle_history_report fuse-list ul li span'):
        t = text(span, default='').lower()
        if 'accident' in t or 'damage' in t:
            history['Accidents or damage'] = 'None reported' if t.startswith('no ') else 'At least 1 reported'
        elif 'owner' in t:
            history['1-owner vehicle'] = 'Yes' if '1 owner' in t else 'No'
        elif 'personal use' in t:
            history['Personal use only'] = 'Yes'
        elif 'recall' in t:
            history['Open recall'] = 'None reported' if 'no ' in t else 'At least 1 reported'
        elif 'title' in t:
            history['Clean Title'] = 'Yes' if 'clean' in t else 'No'
    post['user_history_des'] = history or None

    warranty: dict = {}
    _DASH = {'-', '–', '—'}
    for dt in soup.select('section.sds-page-section.warranty_section dl.fancy-description-list dt'):
        k, v = text(dt), text(dt.find_next_sibling('dd'))
        if k and v: warranty[k] = None if v in _DASH else v
    for dt in soup.select('fuse-popover#cpo-popover dl dt'):
        k = text(dt)
        v = ' '.join((text(dt.find_next_sibling('dd'), default='')).split())
        if k and v: warranty[k] = None if v in _DASH else v
    post['warranty_des'] = warranty or None

    imgs = [
        attr(img, 'src', '').split('?')[0]
        for img in soup.select('fuse-gallery-grid#gallery img')[:5]
        if attr(img, 'src')
    ]
    post['image'] = imgs or None
    if imgs:
        vin = (basics.get('VIN') or basics.get('Stock Number') or 'Unknown')
        download_images(imgs, Path(IMG_BASE_DIR) / 'post_images' / vin)

    return post

print('✅ _parse_post defined')

✅ _parse_post defined


### 6b — Seller

In [9]:
def _parse_seller(soup: BeautifulSoup) -> dict:
    seller: dict = {}
    seller['seller_name'] = text(soup.select_one('div.dealer-info h3'))

    raw_link = attr(soup.select_one('div.dealer-info fuse-stack.rating-stack a'), 'href')
    if raw_link:
        link = absolute_url(raw_link)
        seller['seller_link'] = link
        parts = [p for p in link.split('#')[0].split('/') if p]
        seller['seller_key'] = parts[-1] if parts else None
    else:
        seller['seller_link'] = seller['seller_key'] = None

    seller['destination']    = text(soup.select_one('div.dealer-info div.map-link a'))
    seller['seller_website'] = attr(soup.select_one('div.dealer-info div.website a'), 'href')

    rating_tag = soup.select_one('fuse-stack.rating-stack fuse-rating')
    raw_rating = attr(rating_tag, 'rating')
    seller['seller_rating'] = float(raw_rating) if raw_rating else None

    review_text = text(rating_tag.find('a') if rating_tag else None)
    if review_text:
        m = re.search(r'[\d,]+', review_text)
        seller['seller_rating_count'] = int(m.group(0).replace(',', '')) if m else None
    else:
        seller['seller_rating_count'] = None

    highlights = [text(li) for li in soup.select('div.highlights-tab fuse-list ul li') if text(li)]
    seller['highlights']  = highlights or None
    seller['description'] = text(soup.select_one('div.about-tab__carsons-summary-body cars-line-clamp p'))

    imgs = [
        attr(img, 'src', '').split('?')[0]
        for img in soup.select('card-gallery.dealership-gallery img')[:2]
        if attr(img, 'src')
    ]
    seller['image'] = imgs or None
    if imgs:
        folder_name = seller.get('seller_key') or 'Unknown_Seller'
        download_images(imgs, Path(IMG_BASE_DIR) / 'seller_images' / folder_name)

    return seller


def _parse_dealer_page(soup: BeautifulSoup, data: dict) -> dict:
    out = copy.deepcopy(data)
    phones: dict = {}
    for div in soup.select('div.dealer-phone'):
        kind   = text(div.select_one('span.phone-number-title'), default='Unknown')
        number = text(div.select_one('a.phone-number'))
        if kind and number: phones[kind] = number
    out['seller']['phone_info'] = phones

    hours: dict = {}
    for row in soup.select('table.dealer-hours tr'):
        cells = row.find_all('td')
        if len(cells) == 2:
            hours[text(cells[0], default='').rstrip(':')] = text(cells[1])
    out['seller']['hours'] = hours
    return out

print('✅ _parse_seller / _parse_dealer_page defined')

✅ _parse_seller / _parse_dealer_page defined


### 6c — Car model & reviews

In [10]:
def _parse_car(soup: BeautifulSoup) -> dict:
    car: dict = {}

    rating_text = text(soup.select_one('section#consumer-reviews div.rating-out-of span.rating-count'))
    car['car_rating'] = float(rating_text) if rating_text else None

    rating_div = soup.select_one('.rating-out-of')
    count_text = text(rating_div.find_next_sibling('div') if rating_div else None)
    m = re.search(r'Based on ([\d,]+) reviews', count_text) if count_text else None
    car['car_rating_count'] = int(m.group(1).replace(',', '')) if m else None

    breakdown: dict = {}
    for div in soup.select('section#consumer-reviews div.ratings-breakdown div.review-text'):
        strong = text(div.find('strong'))
        if strong:
            cat = text(div).replace(strong, '').strip()
            try: breakdown[cat] = float(strong)
            except: breakdown[cat] = strong
    car['ratings'] = breakdown or None

    subtitle = text(soup.select_one('section#consumer-reviews div.subtitle'))
    m = re.search(r'(\d+(?:\.\d+)?)%', subtitle) if subtitle else None
    car['percentage_recommend'] = float(m.group(1)) if m else None

    raw_url = attr(soup.select_one('fuse-button.review-btn-secondary[href*="write-a-review"]'), 'href')
    if raw_url:
        full_url = absolute_url(raw_url)
        car['car_link'] = full_url.replace('write-a-review/', '')
        parts = [p for p in full_url.split('/') if p]
        if 'write-a-review' in parts:
            idx  = parts.index('write-a-review')
            slug = parts[idx - 1]
            car['car_model'] = slug
            car['brand']     = slug.split('-')[0].replace('_', ' ').title()
            slug_parts = slug.split('-')
            if slug_parts[-1].isdigit() and len(slug_parts[-1]) == 4:
                year = slug_parts.pop()
                car['car_name'] = f'{year} {" ".join(slug_parts).title()}'
            else:
                car['car_name'] = ' '.join(slug_parts).title()
        car['review_link'] = car['car_link'] + 'consumer-reviews/?page_size=40' if car.get('car_link') else None
    else:
        car.update(car_model=None, car_link=None, review_link=None, brand=None, car_name=None)

    reviews = []
    for block in soup.select('div.consumer-review-container'):
        r: dict = {}
        rating_el = block.select_one('fuse-rating, spark-rating')
        r['overall_rating'] = float(attr(rating_el, 'rating')) if attr(rating_el, 'rating') else None

        time_el = block.select_one('.review-date') or block.select_one('.review-byline > div:nth-child(1)')
        r['time'] = text(time_el)

        author_el = block.select_one('.author-details > div:first-child, .review-byline > div:nth-child(2)')
        byline = text(author_el, default='')
        if byline.startswith('By '): byline = byline[3:]
        if ' from ' in byline:
            name, loc = byline.split(' from ', 1)
            r['user_name'], r['from'] = name.strip(), loc.strip()
        elif ' on ' in byline:
            name, loc = byline.split(' on ', 1)
            r['user_name'], r['from'] = name.strip(), loc.strip()
        else:
            r['user_name'], r['from'] = byline.strip(), None

        r['title'] = text(block.select_one('h3.title'))

        rb: dict = {}
        for stack in block.select('.ratings-breakdown fuse-stack'):
            div = stack.select_one('.review-text')
            sv  = text(div.find('strong') if div else None)
            if sv:
                k = text(div).replace(sv, '').strip()
                try: rb[k] = float(sv)
                except: rb[k] = sv
        if not rb:
            for li in block.select('.review-breakdown--list li'):
                k = text(li.select_one('.sds-definition-list__display-name'))
                v = text(li.select_one('.sds-definition-list__value'))
                if k and v:
                    try: rb[k] = float(v)
                    except: rb[k] = v
        r['ratings_breakdown'] = rb or None

        clamp = block.select_one('cars-line-clamp')
        if clamp:
            fb = clamp.select_one('.review-feedback')
            if fb: fb.extract()
            r['review'] = text(clamp)
        else:
            r['review'] = text(block.select_one('p.review-body'))

        if r.get('review') or r.get('title'):
            reviews.append(r)

    car['reviews'] = reviews or None
    return car

print('✅ _parse_car defined')

✅ _parse_car defined


## Step 7 — Scraping API

In [11]:
def scrape_listing(sb, url: str) -> dict | None:
    """Scrape a single car detail page. Reuses the same sb browser session."""
    soup = get_soup(sb, url, target_css='div.vehicle-details, #vehicle-title, .list-price')
    if soup is None:
        return None
    return {
        'datetime': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'post':   _parse_post(soup),
        'seller': _parse_seller(soup),
        'car':    _parse_car(soup),
    }


def scrape_dealer(sb, url: str, data: dict) -> dict:
    """Enrich data with dealer phone/hours. Returns data unchanged on failure."""
    try:
        soup = get_soup(sb, url, target_css='div.dealer-contact-section, div.dealer-info')
        return _parse_dealer_page(soup, data)
    except Exception as e:
        print(f'  [dealer] {e}')
        return data


def scrape_full(sb, url: str) -> dict | None:
    """Scrape listing + dealer page. Returns merged result."""
    try:
        data = scrape_listing(sb, url)
        if data:
            seller_url = data.get('seller', {}).get('seller_link')
            if seller_url:
                data = scrape_dealer(sb, seller_url, data)
        return data
    except Exception as e:
        print(f'  [scrape_full] {e}')
        return None

print('✅ scrape_listing / scrape_dealer / scrape_full defined')

✅ scrape_listing / scrape_dealer / scrape_full defined


## Step 8 — Link Crawler
> Single browser session across all pages — Cloudflare cookie stays alive.

In [12]:
# def crawl_listing_urls(
#     start_page: int = START_PAGE,
#     end_page: int   = END_PAGE,
#     output_dir: str = LINK_FOLDER,
#     delay: float    = 2.0,
#     headless: bool  = True,
# ) -> None:
#     Path(output_dir).mkdir(parents=True, exist_ok=True)

#     with SB(uc=True, xvfb=True, locale='en',
#             binary_location=CHROME_BINARY) as sb:

#         for page in range(start_page, end_page + 1):
#             url = RESULTS_URL_TMPL.format(page=page)
#             print(f'\n[{page}/{end_page}] {url}')
#             try:
#                 soup = get_soup(sb, url, target_css='a[data-card-link]', scroll=True)
#                 links = [
#                     urljoin(SITE_BASE, tag['href'])
#                     for tag in soup.select('a[data-card-link]')
#                     if tag.get('href') and '/vehicledetail/' in tag['href']
#                 ]
#                 if not links:
#                     print(f'  [!] No links on page {page}')
#                     continue

#                 page_file = Path(output_dir) / f'page_{page}.txt'
#                 page_file.write_text('\n'.join(links) + '\n', encoding='utf-8')
#                 print(f'  ✅ Saved {len(links)} links → {page_file}')

#             except Exception as e:
#                 print(f'  [!] Page {page} error: {e}')
#                 continue

#             sleep_t = delay + random.uniform(0.5, 1.5)
#             print(f'  Sleeping {sleep_t:.1f}s...')
#             time.sleep(sleep_t)

#     print(f'\n✅ Done crawling links → "{output_dir}"')

# print('✅ crawl_listing_urls defined')

## Step 9 — Batch Detail Scraper
> Reads link files, scrapes each car page, saves one JSON per car.
> Safe to resume — skips already-scraped files.

In [13]:
# def scrape_from_files(
#     link_folder: str = LINK_FOLDER,
#     output_root: str = RAW_DATA_DIR,
#     delay: float     = 2.5,
#     headless: bool   = True,
#     from_page: int   = START_PAGE,
#     to_page: int     = END_PAGE,
# ) -> None:
#     root = Path(output_root)
#     root.mkdir(parents=True, exist_ok=True)

#     with SB(uc=True, xvfb=True, locale='en',
#             binary_location=CHROME_BINARY) as sb:

#         for page in range(from_page, to_page + 1):
#             link_file = Path(link_folder) / f'page_{page}.txt'
#             if not link_file.exists():
#                 print(f'[Skip] {link_file} not found — run crawler first')
#                 continue

#             urls     = [u for u in link_file.read_text(encoding='utf-8').splitlines() if u.strip()]
#             page_dir = root / str(page)
#             page_dir.mkdir(exist_ok=True)
#             print(f'\n[Page {page}] {len(urls)} URLs')

#             for i, url in enumerate(urls, 1):
#                 out_file = page_dir / f'{i}.json'
#                 if out_file.exists():
#                     print(f'  [{i}/{len(urls)}] Already done — skipping')
#                     continue

#                 print(f'  [{i}/{len(urls)}] {url}')
#                 try:
#                     data = scrape_full(sb, url)
#                     if data:
#                         out_file.write_text(
#                             json.dumps(data, ensure_ascii=False, indent=2),
#                             encoding='utf-8'
#                         )
#                         print(f'    ✅ Saved → {out_file}')
#                     else:
#                         print(f'    [!] No data returned')
#                 except Exception as e:
#                     print(f'    [!] {e}')

#                 time.sleep(delay + random.uniform(0.5, 1.5))

#     print('\n✅ All done.')

# print('✅ scrape_from_files defined')

## Step 10 — GCS Upload *(Colab only, optional)*

In [14]:
def upload_to_gcs(from_page: int = START_PAGE, to_page: int = END_PAGE) -> None:
    try:
        from google.cloud import storage
    except ImportError:
        print('[GCS] Install: pip install google-cloud-storage')
        return

    client = storage.Client()
    bucket = client.bucket(GCS_BUCKET)

    print('Uploading JSON files...')
    json_ok = json_fail = 0
    for page in range(from_page, to_page + 1):
        page_dir = Path(RAW_DATA_DIR) / str(page)
        if not page_dir.is_dir():
            print(f'  [Skip] {page_dir} not found')
            continue
        for f in sorted(page_dir.glob('*.json')):
            blob_name = f'{GCS_PREFIX}/{page}/{f.name}'
            try:
                bucket.blob(blob_name).upload_from_filename(
                    str(f), content_type='application/json'
                )
                json_ok += 1
            except Exception as e:
                print(f'  [!] {blob_name}: {e}')
                json_fail += 1
    print(f'  ✅ JSON: {json_ok} uploaded, {json_fail} failed')

    print('\n Uploading images...')
    img_ok = img_fail = 0

    img_base = Path(IMG_BASE_DIR)
    if not img_base.exists():
        print('  [Skip] No downloaded_images folder found')
    else:
        for img_file in img_base.rglob('*.jpg'):

            relative = img_file.relative_to(img_base)          # e.g. post_images/VIN123/1.jpg
            blob_name = f'images/{relative}'                    # e.g. images/post_images/VIN123/1.jpg
            try:
                bucket.blob(blob_name).upload_from_filename(
                    str(img_file), content_type='image/jpeg'
                )
                img_ok += 1
            except Exception as e:
                print(f'  [!] {blob_name}: {e}')
                img_fail += 1

        print(f'  ✅ Images: {img_ok} uploaded, {img_fail} failed')

    print(f'\n GCS upload complete')
    print(f'   JSON  → gs://{GCS_BUCKET}/{GCS_PREFIX}/')
    print(f'   Images → gs://{GCS_BUCKET}/images/')

print('✅ upload_to_gcs defined')

✅ upload_to_gcs defined


---
## Run — Step 1: Crawl Listing URLs
Collects vehicle URLs from Cars.com search pages → `car_links/page_N.txt`

In [15]:
# crawl_listing_urls(
#     start_page = START_PAGE,
#     end_page   = END_PAGE,
#     output_dir = LINK_FOLDER,
#     delay      = 2.0,
#     headless   = True,
# )

## Run — Step 2: Scrape Car Detail Pages
Reads link files → scrapes each car → saves `raw_data/<page>/<index>.json`

In [16]:
# scrape_from_files(
#     link_folder = LINK_FOLDER,
#     output_root = RAW_DATA_DIR,
#     delay       = 2.5,
#     headless    = True,
#     from_page   = START_PAGE,
#     to_page     = END_PAGE,
# )

## Run — Step 3 : Upload to GCS
Authenticate first: `from google.colab import auth; auth.authenticate_user()`

In [17]:
# from google.colab import auth
# auth.authenticate_user()
# upload_to_gcs(from_page=START_PAGE, to_page=END_PAGE)

#optimize

In [18]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from queue import Queue, Empty

MAX_BROWSER_WORKERS = 5
RETRY_LIMIT         = 3
INTER_REQUEST_DELAY = 1.5

def _scrape_worker(
    worker_id: int,
    urls_with_index: list[tuple[int, str]],
    page_dir: Path,
    results: dict,
    lock: threading.Lock,
) -> None:
    done = fail = skip = 0

    with SB(uc=True, xvfb=True, locale='en',
            binary_location=CHROME_BINARY) as sb:

        for idx, url in urls_with_index:
            out_file = page_dir / f'{idx}.json'

            if out_file.exists():
                with lock:
                    print(f'  [W{worker_id}] [{idx}] Skip (already done)')
                skip += 1
                continue

            success = False
            for attempt in range(1, RETRY_LIMIT + 1):
                try:
                    with lock:
                        print(f'  [W{worker_id}] [{idx}/{attempt}] {url}')

                    data = scrape_full(sb, url)

                    if data:
                        out_file.write_text(
                            json.dumps(data, ensure_ascii=False, indent=2),
                            encoding='utf-8'
                        )
                        with lock:
                            print(f'  [W{worker_id}] [{idx}] ✅ Saved')
                        done += 1
                        success = True
                        break
                    else:
                        with lock:
                            print(f'  [W{worker_id}] [{idx}] No data — retry {attempt}/{RETRY_LIMIT}')

                except Exception as e:
                    with lock:
                        print(f'  [W{worker_id}] [{idx}] Error (attempt {attempt}): {e}')
                    time.sleep(random.uniform(2, 4))

            if not success:
                fail += 1
                with lock:
                    print(f'  [W{worker_id}] [{idx}] ❌ Failed after {RETRY_LIMIT} attempts')

            time.sleep(INTER_REQUEST_DELAY + random.uniform(0.3, 1.0))

    with lock:
        results[worker_id] = {'done': done, 'fail': fail, 'skip': skip}
        print(f'\n[W{worker_id}] Finished — done={done}, fail={fail}, skip={skip}')


def _split_urls(urls_with_index: list, n_workers: int) -> list[list]:
    buckets = [[] for _ in range(n_workers)]
    for i, item in enumerate(urls_with_index):
        buckets[i % n_workers].append(item)
    return buckets


def scrape_from_files_parallel(
    link_folder: str  = LINK_FOLDER,
    output_root: str  = RAW_DATA_DIR,
    from_page: int    = START_PAGE,
    to_page: int      = END_PAGE,
    n_workers: int    = MAX_BROWSER_WORKERS,
    delay_between_pages: float = 2.0,
) -> None:
    root = Path(output_root)
    root.mkdir(parents=True, exist_ok=True)
    lock = threading.Lock()

    total_done = total_fail = total_skip = 0

    for page in range(from_page, to_page + 1):
        link_file = Path(link_folder) / f'page_{page}.txt'
        if not link_file.exists():
            print(f'[Skip] {link_file} not found')
            continue

        urls = [u.strip() for u in link_file.read_text(encoding='utf-8').splitlines() if u.strip()]
        page_dir = root / str(page)
        page_dir.mkdir(exist_ok=True)

        pending = [(i, u) for i, u in enumerate(urls, 1)
                   if not (page_dir / f'{i}.json').exists()]
        already = len(urls) - len(pending)

        print(f'\n{"="*60}')
        print(f'[Page {page}] Total: {len(urls)} | Pending: {len(pending)} | Already done: {already}')
        print(f'[Page {page}] Using {min(n_workers, len(pending))} workers')
        print(f'{"="*60}')

        if not pending:
            print(f'  All URLs already scraped — skipping page {page}')
            continue

        actual_workers = min(n_workers, len(pending))
        buckets        = _split_urls(pending, actual_workers)
        results        = {}

        threads = []
        for wid, bucket in enumerate(buckets):
            t = threading.Thread(
                target=_scrape_worker,
                args=(wid + 1, bucket, page_dir, results, lock),
                daemon=True,
            )
            threads.append(t)
            t.start()
            time.sleep(random.uniform(3, 6))

        for t in threads:
            t.join()

        page_done = sum(r['done'] for r in results.values())
        page_fail = sum(r['fail'] for r in results.values())
        page_skip = sum(r['skip'] for r in results.values())
        total_done += page_done
        total_fail += page_fail
        total_skip += page_skip

        print(f'\n[Page {page}] ✅ Done={page_done} | ❌ Failed={page_fail} | ⏭ Skipped={page_skip}')

        if page < to_page:
            print(f'Waiting {delay_between_pages}s before next page...')
            time.sleep(delay_between_pages)

    print(f'\n{"="*60}')
    print(f'🏁 ALL DONE')
    print(f'   ✅ Scraped : {total_done}')
    print(f'   ❌ Failed  : {total_fail}')
    print(f'   ⏭ Skipped : {total_skip}')
    print(f'{"="*60}')


print('✅ scrape_from_files_parallel defined')

✅ scrape_from_files_parallel defined


In [19]:
def crawl_listing_urls_robust(
    start_page: int = START_PAGE,
    end_page: int   = END_PAGE,
    output_dir: str = LINK_FOLDER,
    delay: float    = 2.0,
    max_retries: int = 3,
) -> None:
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    with SB(uc=True, xvfb=True, locale='en',
            binary_location=CHROME_BINARY) as sb:

        for page in range(start_page, end_page + 1):
            page_file = Path(output_dir) / f'page_{page}.txt'

            if page_file.exists() and page_file.stat().st_size > 0:
                existing = page_file.read_text().strip().splitlines()
                print(f'[{page}/{end_page}] ⏭ Already have {len(existing)} links — skipping')
                continue

            url     = RESULTS_URL_TMPL.format(page=page)
            success = False

            for attempt in range(1, max_retries + 1):
                try:
                    print(f'\n[{page}/{end_page}] Attempt {attempt}/{max_retries}: {url}')
                    soup = get_soup(sb, url, target_css='a[data-card-link]', scroll=True)

                    links = [
                        urljoin(SITE_BASE, tag['href'])
                        for tag in soup.select('a[data-card-link]')
                        if tag.get('href') and '/vehicledetail/' in tag['href']
                    ]

                    if not links:
                        raise ValueError('No links found — possible block')

                    page_file.write_text('\n'.join(links) + '\n', encoding='utf-8')
                    print(f'  ✅ Saved {len(links)} links → {page_file}')
                    success = True
                    break

                except Exception as e:
                    wait = (2 ** attempt) + random.uniform(1, 3)
                    print(f'  [!] {e}')
                    if attempt < max_retries:
                        print(f'  Backoff {wait:.1f}s before retry...')
                        time.sleep(wait)

            if not success:
                print(f'  ❌ Page {page} failed after {max_retries} attempts — continuing')

            time.sleep(delay + random.uniform(0.5, 1.5))

    print(f'\n✅ Done. Links saved to "{output_dir}"')


print('✅ crawl_listing_urls_robust defined')

✅ crawl_listing_urls_robust defined


In [24]:
crawl_listing_urls_robust(
    start_page  = START_PAGE,
    end_page    = END_PAGE,
    output_dir  = LINK_FOLDER,
    delay       = 2.0,
    max_retries = 3,
)


[41/50] Attempt 1/3: https://www.cars.com/shopping/results/?deal_ratings%5B%5D=good&zip=60606&maximum_distance=9999&sort=best_match_desc&page=41
  ✅ Saved 24 links → car_links/page_41.txt

[42/50] Attempt 1/3: https://www.cars.com/shopping/results/?deal_ratings%5B%5D=good&zip=60606&maximum_distance=9999&sort=best_match_desc&page=42
  ✅ Saved 24 links → car_links/page_42.txt

[43/50] Attempt 1/3: https://www.cars.com/shopping/results/?deal_ratings%5B%5D=good&zip=60606&maximum_distance=9999&sort=best_match_desc&page=43
  ✅ Saved 24 links → car_links/page_43.txt

[44/50] Attempt 1/3: https://www.cars.com/shopping/results/?deal_ratings%5B%5D=good&zip=60606&maximum_distance=9999&sort=best_match_desc&page=44
  ✅ Saved 24 links → car_links/page_44.txt

[45/50] Attempt 1/3: https://www.cars.com/shopping/results/?deal_ratings%5B%5D=good&zip=60606&maximum_distance=9999&sort=best_match_desc&page=45
  ✅ Saved 24 links → car_links/page_45.txt

[46/50] Attempt 1/3: https://www.cars.com/shopping/res

In [ ]:
scrape_from_files_parallel(
    link_folder = LINK_FOLDER,
    output_root = RAW_DATA_DIR,
    from_page   = START_PAGE,
    to_page     = END_PAGE,
    n_workers   = 5,
)


[Page 41] Total: 24 | Pending: 24 | Already done: 0
[Page 41] Using 5 workers
  [W1] [1/1] https://www.cars.com/vehicledetail/a0054d7d-cf44-41a7-9060-faaf8fb65256/?attribution_type=p_one
  [W2] [2/1] https://www.cars.com/vehicledetail/c0c60435-9e4f-4d3b-9cb4-74ff97b7d792/?attribution_type=isa
  [captcha] Turnstile detected — solving...
  [captcha] Turnstile detected — solving...
  [W3] [3/1] https://www.cars.com/vehicledetail/b46d34f1-b365-4203-bb02-93bc538292d1/
  [W4] [4/1] https://www.cars.com/vehicledetail/cac40598-9cc7-4497-b27a-fde383e1b07d/
  [captcha] ✅ Solved
  [captcha] Turnstile detected — solving...
  [W5] [5/1] https://www.cars.com/vehicledetail/14698127-303c-48e2-a141-01ec15ba09f3/
  [captcha] Turnstile detected — solving...
  [captcha] ✅ Solved
  [W1] [1] ✅ Saved
  [W1] [6/1] https://www.cars.com/vehicledetail/2921076d-814c-4718-a663-a580f9997db3/
  [captcha] Turnstile detected — solving...
  [captcha] ✅ Solved
  [captcha] ✅ Solved
  [W3] [3] ✅ Saved
  [captcha] ✅ Solve

In [ ]:
upload_to_gcs(from_page=START_PAGE, to_page=END_PAGE)